In [1]:
import pandas as pd
import numpy as np

print("Environment is working")
print(pd.__version__)
print(np.__version__)

Environment is working
3.0.1
2.4.3


In [6]:
from pathlib import Path

raw_path = Path("/Users/dhanushgarikapati/olist-end-to-end-analysis/data/raw")

files = list(raw_path.glob("*.csv"))
files

[PosixPath('/Users/dhanushgarikapati/olist-end-to-end-analysis/data/raw/olist_sellers_dataset.csv'),
 PosixPath('/Users/dhanushgarikapati/olist-end-to-end-analysis/data/raw/product_category_name_translation.csv'),
 PosixPath('/Users/dhanushgarikapati/olist-end-to-end-analysis/data/raw/olist_orders_dataset.csv'),
 PosixPath('/Users/dhanushgarikapati/olist-end-to-end-analysis/data/raw/olist_order_items_dataset.csv'),
 PosixPath('/Users/dhanushgarikapati/olist-end-to-end-analysis/data/raw/olist_customers_dataset.csv'),
 PosixPath('/Users/dhanushgarikapati/olist-end-to-end-analysis/data/raw/olist_geolocation_dataset.csv'),
 PosixPath('/Users/dhanushgarikapati/olist-end-to-end-analysis/data/raw/olist_order_payments_dataset.csv'),
 PosixPath('/Users/dhanushgarikapati/olist-end-to-end-analysis/data/raw/olist_order_reviews_dataset.csv'),
 PosixPath('/Users/dhanushgarikapati/olist-end-to-end-analysis/data/raw/olist_products_dataset.csv')]

In [7]:
summary = []

for file in files:
    df = pd.read_csv(file)
    summary.append({
        "file_name": file.name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "column_names": ", ".join(df.columns.tolist())
    })

summary_df = pd.DataFrame(summary)
summary_df

,file_name,rows,columns,column_names
0,olist_sellers_dataset.csv,3095,4,"seller_id, seller_zip_code_prefix, seller_city..."
1,product_category_name_translation.csv,71,2,"product_category_name, product_category_name_e..."
2,olist_orders_dataset.csv,99441,8,"order_id, customer_id, order_status, order_pur..."
3,olist_order_items_dataset.csv,112650,7,"order_id, order_item_id, product_id, seller_id..."
4,olist_customers_dataset.csv,99441,5,"customer_id, customer_unique_id, customer_zip_..."
5,olist_geolocation_dataset.csv,1000163,5,"geolocation_zip_code_prefix, geolocation_lat, ..."
6,olist_order_payments_dataset.csv,103886,5,"order_id, payment_sequential, payment_type, pa..."
7,olist_order_reviews_dataset.csv,99224,7,"review_id, order_id, review_score, review_comm..."
8,olist_products_dataset.csv,32951,9,"product_id, product_category_name, product_nam..."


In [8]:
for file in files:
    df = pd.read_csv(file)
    print("=" * 80)
    print(f"FILE: {file.name}")
    print(df.head())
    print("\nINFO:")
    print(df.info())
    print("\nNULL COUNTS:")
    print(df.isnull().sum())
    print("\nDUPLICATES:", df.duplicated().sum())

FILE: olist_sellers_dataset.csv
                          seller_id  seller_zip_code_prefix  \
0  3442f8959a84dea7ee197c632cb2df15                   13023   
1  d1b65fc7debc3361ea86b5f14c68d2e2                   13844   
2  ce3ad9de960102d0677a81f5d0bb7b2d                   20031   
3  c0f3eea2e14555b6faeea3dd58c1b1c3                    4195   
4  51a04a8a6bdcb23deccc82b0b80742cf                   12914   

         seller_city seller_state  
0           campinas           SP  
1         mogi guacu           SP  
2     rio de janeiro           RJ  
3          sao paulo           SP  
4  braganca paulista           SP  

INFO:
<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_stat

In [10]:
audit_notes = pd.DataFrame([
    {
        "file": "olist_orders_dataset.csv",
        "grain": "one row per order",
        "primary_key": "order_id",
        "join_keys": "customer_id, order_id",
        "date_columns": "order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date",
        "notes": "central order lifecycle table; delivery-related dates can be null for non-delivered/non-finalized orders"
    },
    {
        "file": "olist_order_items_dataset.csv",
        "grain": "one row per order item",
        "primary_key": "order_id + order_item_id",
        "join_keys": "order_id, product_id, seller_id",
        "date_columns": "shipping_limit_date",
        "notes": "transaction line-item table; contains item price and freight value"
    },
    {
        "file": "olist_order_payments_dataset.csv",
        "grain": "one row per payment sequence within an order",
        "primary_key": "order_id + payment_sequential",
        "join_keys": "order_id",
        "date_columns": "None",
        "notes": "payment table; multiple payment rows can exist for a single order"
    },
    {
        "file": "olist_order_reviews_dataset.csv",
        "grain": "one row per review record",
        "primary_key": "review_id (verify uniqueness later)",
        "join_keys": "order_id",
        "date_columns": "review_creation_date, review_answer_timestamp",
        "notes": "review text fields are heavily null; review_score is fully populated"
    },
    {
        "file": "olist_customers_dataset.csv",
        "grain": "one row per customer_id",
        "primary_key": "customer_id",
        "join_keys": "customer_id, customer_unique_id",
        "date_columns": "None",
        "notes": "use customer_unique_id for customer-level analysis and repeat customer metrics"
    },
    {
        "file": "olist_products_dataset.csv",
        "grain": "one row per product",
        "primary_key": "product_id",
        "join_keys": "product_id, product_category_name",
        "date_columns": "None",
        "notes": "some category and product metadata are missing"
    },
    {
        "file": "olist_sellers_dataset.csv",
        "grain": "one row per seller",
        "primary_key": "seller_id",
        "join_keys": "seller_id",
        "date_columns": "None",
        "notes": "seller location dimension"
    },
    {
        "file": "product_category_name_translation.csv",
        "grain": "one row per product category translation",
        "primary_key": "product_category_name",
        "join_keys": "product_category_name",
        "date_columns": "None",
        "notes": "maps Portuguese category labels to English"
    },
    {
        "file": "olist_geolocation_dataset.csv",
        "grain": "multiple geolocation reference rows by zip prefix",
        "primary_key": "None",
        "join_keys": "geolocation_zip_code_prefix",
        "date_columns": "None",
        "notes": "contains many duplicates; deduplicate or aggregate before joining"
    }
])

audit_notes

,file,grain,primary_key,join_keys,date_columns,notes
0,olist_orders_dataset.csv,one row per order,order_id,"customer_id, order_id","order_purchase_timestamp, order_approved_at, o...",central order lifecycle table; delivery-relate...
1,olist_order_items_dataset.csv,one row per order item,order_id + order_item_id,"order_id, product_id, seller_id",shipping_limit_date,transaction line-item table; contains item pri...
2,olist_order_payments_dataset.csv,one row per payment sequence within an order,order_id + payment_sequential,order_id,None,payment table; multiple payment rows can exist...
3,olist_order_reviews_dataset.csv,one row per review record,review_id (verify uniqueness later),order_id,"review_creation_date, review_answer_timestamp",review text fields are heavily null; review_sc...
4,olist_customers_dataset.csv,one row per customer_id,customer_id,"customer_id, customer_unique_id",None,use customer_unique_id for customer-level anal...
5,olist_products_dataset.csv,one row per product,product_id,"product_id, product_category_name",None,some category and product metadata are missing
6,olist_sellers_dataset.csv,one row per seller,seller_id,seller_id,None,seller location dimension
7,product_category_name_translation.csv,one row per product category translation,product_category_name,product_category_name,None,maps Portuguese category labels to English
8,olist_geolocation_dataset.csv,multiple geolocation reference rows by zip prefix,None,geolocation_zip_code_prefix,None,contains many duplicates; deduplicate or aggre...
